## 🔒 Estrutura do notebook e importações:

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA

#------------------------------------------------------------
# Backtest Walk-Forward — Finanças Quantitativas
#------------------------------------------------------------

#┌─────────────────────────────────────────────────────┐
#│  CÉLULAS 1-2   INPUT DOS DADOS - EDITAR             │
#│                (Ativo - Período - Setup - Custos)   │
#├─────────────────────────────────────────────────────┤
#│  CÉLULA 3      Parâmetros do Walk_Forward           │
#│                (Janela de Treino e Teste)           │
#├─────────────────────────────────────────────────────┤
#│  CÉLULAS 4-9   FUNÇÕES (NÃO EDITAR)                 │
#│                (Custos-Métricas-WF-Estatísticas     │
#├─────────────────────────────────────────────────────┤
#│  CÉLULA 10     RODA TUDO E PLOTA OS GRÁFICOS        │
#└─────────────────────────────────────────────────────┘
#------------------------------------------------------------
## Como usar este notebook
#------------------------------------------------------------
#1. Execute as células **de cima para baixo** clicando no ▶ de cada uma
#2. **Não altere** as células marcadas com 🔒
#3. As únicas 2 células que seu grupo deve editar estão marcadas com ✏️
#4. No final, execute a célula 13 para ver todos os resultados

#------------------------------------------------------------
# Importação das bibliotecas necessárias para o notebook inteiro
#------------------------------------------------------------
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

## CÉLULA 1 — ✏️ [INPUT] ATIVO - PERÍODO DO HISTÓRICO - DATASET

In [ ]:
#----------------------------------------------------------
#  ✏️ INPUT DO CÓDIGO - ATIVO E HISTÓRICO  (adaptado para o artigo)
#----------------------------------------------------------
# Lê o dataset sintético Market_Trend_External (Kaggle) em vez de baixar
# um ativo do yfinance. O notebook descarta o último ano carregado, por isso
# os dados vão até 1936: assim o último ano de teste é 1935.

ticker       = 'Market_Trend_External (sintético)'
Data_Inicial = '1924-01-01'
Data_Final   = '1936-12-31'

data = pd.read_csv('../dados/Market_Trend_External.csv', parse_dates=['Date'])
data = data.rename(columns={'Close_Price': 'Close'})
data = data[(data['Date'] >= Data_Inicial) & (data['Date'] <= Data_Final)].reset_index(drop=True)

# Cria coluna com o ano de cada pregão (usada no walk-forward)
data['Year'] = data['Date'].dt.year

# Lista ordenada de todos os anos disponíveis
years = sorted(data['Year'].unique())

print(f'Ativo      : {ticker}')
print(f'Período    : {Data_Inicial} → {Data_Final}')
print(f'Pregões    : {len(data)}')
print(f'Anos       : {years[0]} → {years[-1]}')

## CÉLULA 2 — ✏️ [INPUT] SETUP DO GRUPO — EDITE AQUI

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║   ✏️  CÉLULA DO GRUPO — EDITE O SETUP APENAS AQUI          ║
# ║                                                             ║
# ║   Todas as outras células já estão prontas.                 ║
# ║   Seu grupo só precisa preencher as seções abaixo.          ║
# ║   Siga os exemplos e comentários com atenção.               ║
# ╚═════════════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd


# ================================================================
# SEÇÃO A — IDENTIFICAÇÃO DO GRUPO
# ================================================================
# Preencha os três campos abaixo com as informações do seu grupo.
# Esses dados aparecerão nos títulos dos gráficos e relatórios.

NOME_DO_GRUPO = 'Grupo 01'          # ex: 'Grupo 03'
ATIVO         = 'PETR4.SA'          # ex: 'VALE3.SA', 'ITUB4.SA'
DESCRICAO     = 'Médias Móveis'     # ex: 'RSI', 'Bollinger + Stop 5%'


# ================================================================
# SEÇÃO B — CUSTO DE TRADE
# ================================================================
# Custo cobrado a cada vez que a estratégia ENTRA ou SAI do mercado.
# 0.003 significa 0.3% por operação.
# Um trade completo (entrar + sair) custa o dobro: ~0.6% no total.
#
# Não altere este valor a menos que o professor oriente diferente.

CUSTO_TRADE = 0.003


# ================================================================
# SEÇÃO C — GRADE DE PARÂMETROS
# ================================================================
#
# ┌─────────────────────────────────────────────────────────────┐
# │ O QUE É A GRADE?                                            │
# │                                                             │
# │ A grade é a lista de todas as combinações de parâmetros     │
# │ que o walk-forward vai TESTAR no período de treino.         │
# │                                                             │
# │ O sistema vai testar cada combinação, calcular o Sharpe,    │
# │ e usar a MELHOR combinação para operar no período de teste. │
# └─────────────────────────────────────────────────────────────┘
#
# REGRA ESSENCIAL:
#   Os nomes das chaves que você usar aqui (ex: 'periodo', 'fator')
#   DEVEM ser os mesmos que você usar na função calcular_sinal()
#   logo abaixo. O sistema passa o dicionário diretamente.
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 1 — Setup com 1 parâmetro (ex: período do RSI)
#
#   def gerar_grade():
#       grade = []
#       for periodo in range(5, 21):        # testa períodos de 5 a 20
#           grade.append({'periodo': periodo})
#       return grade
#
#   → Isso cria 16 combinações:
#     [{'periodo': 5}, {'periodo': 6}, ..., {'periodo': 20}]
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 2 — Setup com 2 parâmetros (ex: RSI + nível de entrada)
#
#   def gerar_grade():
#       grade = []
#       for periodo in range(5, 21):
#           for nivel in [25, 30, 35]:      # 3 níveis de entrada
#               grade.append({
#                   'periodo'      : periodo,
#                   'rsi_entrada'  : nivel
#               })
#       return grade
#
#   → Isso cria 16 × 3 = 48 combinações
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 3 — Setup com 3 parâmetros (ex: Bollinger com stop)
#
#   def gerar_grade():
#       grade = []
#       for periodo in [10, 15, 20]:
#           for desvios in [1.5, 2.0, 2.5]:
#               for stop in [0.03, 0.05, 0.08]:   # 3%, 5%, 8%
#                   grade.append({
#                       'periodo'   : periodo,
#                       'desvios'   : desvios,
#                       'stop_loss' : stop
#                   })
#       return grade
#
#   → Isso cria 3 × 3 × 3 = 27 combinações
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 4 — Setup com 2 indicadores (MA curta + MA longa)
#
#   def gerar_grade():
#       grade = []
#       for short in range(3, 16):          # MA curta: 3 a 15 dias
#           for long_ in range(30, 71):     # MA longa: 30 a 70 dias
#               grade.append({
#                   'short' : short,
#                   'long'  : long_
#               })
#       return grade
#
#   → Isso cria 13 × 41 = 533 combinações
#
# ⚠️  ATENÇÃO: quanto mais combinações, mais tempo o walk-forward
#    demora. Acima de 1.000 combinações pode demorar vários minutos.
# ────────────────────────────────────────────────────────────────

def gerar_grade():
    """
    EDITE O CORPO DESTA FUNÇÃO com os parâmetros do seu setup.
    Retorne sempre uma lista de dicionários.
    """

    grade = []

    # ---- ESCREVA SUA GRADE AQUI (apague e substitua o exemplo) ----

    for short in range(3, 16):
        for long_ in range(30, 71):
            grade.append({
                'short' : short,
                'long'  : long_
            })

    # ---------------------------------------------------------------

    return grade   # ← não apague esta linha


# ================================================================
# SEÇÃO D — LÓGICA DA ESTRATÉGIA (calcular_sinal)
# ================================================================
#
# ┌─────────────────────────────────────────────────────────────┐
# │ O QUE ESTA FUNÇÃO FAZ?                                      │
# │                                                             │
# │ Para cada dia, ela decide: o grupo está DENTRO (1)          │
# │ ou FORA (0) do mercado?                                     │
# │                                                             │
# │ Ela recebe uma tabela de preços ('janela') e um conjunto    │
# │ de parâmetros ('params'), e devolve uma coluna de 0s e 1s.  │
# └─────────────────────────────────────────────────────────────┘
#
# A FUNÇÃO RECEBE:
#
#   janela  → tabela de preços com as colunas:
#               'Close'  : preço de fechamento  ← mais usado
#               'Open'   : preço de abertura
#               'High'   : máxima do dia
#               'Low'    : mínima do dia
#               'Volume' : volume negociado
#
#   params  → dicionário com os parâmetros desta combinação.
#             Os nomes das chaves são os mesmos que você definiu
#             em gerar_grade(). Acesse com params['nome_chave'].
#             Exemplos:
#               params['short']       → valor da MA curta
#               params['periodo']     → período do RSI
#               params['stop_loss']   → nível de stop
#
# A FUNÇÃO DEVE RETORNAR:
#
#   Uma pd.Series com valores 0 ou 1 para cada linha da janela:
#     1 = comprado (estamos dentro do mercado naquele dia)
#     0 = fora do mercado (sem posição)
#
# REGRAS IMPORTANTES:
#   ✓ Use APENAS dados da 'janela' recebida (sem look-ahead)
#   ✓ Retorne APENAS 0 ou 1 (sem valores intermediários)
#   ✓ Use index=janela.index ao criar a Series final
#   ✗ NÃO use .shift() aqui — o sistema já aplica automaticamente
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 1 — 1 indicador, 1 parâmetro: RSI simples
#
#   Lógica: comprado quando RSI < 30 (sobrevendido)
#           fora quando RSI ≥ 30
#
#   def calcular_sinal(janela, params):
#       delta = janela['Close'].diff()
#       ganho = delta.clip(lower=0).rolling(params['periodo']).mean()
#       perda = (-delta.clip(upper=0)).rolling(params['periodo']).mean()
#       rsi   = 100 - (100 / (1 + ganho / perda))
#       sinal = pd.Series(
#           np.where(rsi < params['rsi_entrada'], 1, 0),
#           index=janela.index
#       )
#       return sinal
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 2 — 1 indicador, 2 parâmetros: RSI com entrada e saída
#
#   Lógica: entra quando RSI < rsi_entrada (sobrevendido)
#           sai   quando RSI > rsi_saida   (sobrecomprado)
#           mantém posição nos dias intermediários
#
#   def calcular_sinal(janela, params):
#       delta = janela['Close'].diff()
#       ganho = delta.clip(lower=0).rolling(params['periodo']).mean()
#       perda = (-delta.clip(upper=0)).rolling(params['periodo']).mean()
#       rsi   = 100 - (100 / (1 + ganho / perda))
#       sinal = pd.Series(
#           np.where(rsi < params['rsi_entrada'], 1,       # entra
#           np.where(rsi > params['rsi_saida'],   0,       # sai
#                                                 np.nan)),# mantém
#           index=janela.index
#       ).ffill().fillna(0).astype(int)
#       return sinal
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 3 — 2 indicadores: cruzamento de médias móveis
#
#   Lógica: comprado quando MA curta > MA longa (tendência de alta)
#           fora    quando MA curta ≤ MA longa (tendência de baixa)
#
#   def calcular_sinal(janela, params):
#       ma_curta = janela['Close'].rolling(params['short']).mean()
#       ma_longa = janela['Close'].rolling(params['long']).mean()
#       sinal = pd.Series(
#           np.where(ma_curta > ma_longa, 1, 0),
#           index=janela.index
#       )
#       return sinal
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 4 — 2 indicadores: Bollinger Bands
#
#   Lógica: entra quando preço toca a banda inferior (oversold)
#           sai   quando preço toca a banda superior (overbought)
#           mantém posição nos dias intermediários
#
#   def calcular_sinal(janela, params):
#       media = janela['Close'].rolling(params['periodo']).mean()
#       std   = janela['Close'].rolling(params['periodo']).std()
#       lower = media - params['desvios'] * std
#       upper = media + params['desvios'] * std
#       sinal = pd.Series(
#           np.where(janela['Close'] < lower, 1,       # entra
#           np.where(janela['Close'] > upper, 0,       # sai
#                                             np.nan)),# mantém
#           index=janela.index
#       ).ffill().fillna(0).astype(int)
#       return sinal
#
# ────────────────────────────────────────────────────────────────
# EXEMPLO 5 — 3 indicadores: MA + RSI + Volume
#
#   Lógica: comprado quando MA curta > MA longa
#           E RSI < 70 (não sobrecomprado)
#           E volume hoje > média de volume dos últimos N dias
#
#   def calcular_sinal(janela, params):
#       ma_curta = janela['Close'].rolling(params['short']).mean()
#       ma_longa = janela['Close'].rolling(params['long']).mean()
#
#       delta = janela['Close'].diff()
#       ganho = delta.clip(lower=0).rolling(params['rsi_p']).mean()
#       perda = (-delta.clip(upper=0)).rolling(params['rsi_p']).mean()
#       rsi   = 100 - (100 / (1 + ganho / perda))
#
#       vol_media = janela['Volume'].rolling(params['vol_p']).mean()
#
#       condicao = (
#           (ma_curta > ma_longa) &       # tendência de alta
#           (rsi < 70) &                  # não sobrecomprado
#           (janela['Volume'] > vol_media) # volume acima da média
#       )
#       sinal = pd.Series(
#           np.where(condicao, 1, 0),
#           index=janela.index
#       )
#       return sinal
# ────────────────────────────────────────────────────────────────

def calcular_sinal(janela, params):
    """
    EDITE O CORPO DESTA FUNÇÃO com a lógica da sua estratégia.
    Retorne sempre uma pd.Series de 0s e 1s com index=janela.index.
    """

    # ---- ESCREVA SUA LÓGICA AQUI (apague e substitua o exemplo) ----

    ma_curta = janela['Close'].rolling(params['short']).mean()
    ma_longa = janela['Close'].rolling(params['long']).mean()

    sinal = pd.Series(
        np.where(ma_curta > ma_longa, 1, 0),
        index=janela.index
    )

    # ----------------------------------------------------------------

    return sinal   # ← não apague esta linha


# ================================================================
# SEÇÃO E — STOP LOSS E STOP GAIN (opcional)
# ================================================================
#
# ┌─────────────────────────────────────────────────────────────┐
# │ O QUE É UM STOP?                                            │
# │                                                             │
# │ Stop loss:  sai do mercado automaticamente quando a         │
# │             perda acumulada desde a entrada atinge X%.      │
# │             Protege contra perdas grandes.                  │
# │                                                             │
# │ Stop gain:  sai do mercado automaticamente quando o         │
# │             lucro acumulado desde a entrada atinge Y%.      │
# │             Garante o lucro alcançado.                      │
# └─────────────────────────────────────────────────────────────┘
#
# ── SE SEU GRUPO NÃO USA STOP ──────────────────────────────────
#
#   Não faça nada. Deixe a função exatamente como está.
#   A linha "if stop_loss is None and stop_gain is None"
#   vai retornar sem modificar nada.
#
# ── SE SEU GRUPO USA STOP LOSS ─────────────────────────────────
#
#   Passo 1: adicione 'stop_loss' na sua grade:
#
#     grade.append({
#         'short'     : short,
#         'long'      : long_,
#         'stop_loss' : 0.05     # ← stop loss de 5%
#     })
#
#   Passo 2: a função aplicar_stop() abaixo já está implementada.
#            Não precisa alterar nada — ela lê 'stop_loss'
#            automaticamente do dicionário params.
#
#   Exemplo: stop_loss = 0.05 significa que se o trade acumular
#            5% de perda desde a entrada, saímos imediatamente.
#
# ── SE SEU GRUPO USA STOP GAIN ─────────────────────────────────
#
#   Passo 1: adicione 'stop_gain' na sua grade:
#
#     grade.append({
#         'short'     : short,
#         'long'      : long_,
#         'stop_gain' : 0.08     # ← stop gain de 8%
#     })
#
#   Passo 2: a função aplicar_stop() abaixo já está implementada.
#            Ela lê 'stop_gain' automaticamente.
#
# ── SE SEU GRUPO USA OS DOIS ───────────────────────────────────
#
#   Passo 1: adicione os dois na grade:
#
#     grade.append({
#         'short'     : short,
#         'long'      : long_,
#         'stop_loss' : 0.05,    # sai se perder 5%
#         'stop_gain' : 0.10     # sai se ganhar 10%
#     })
#
#   Passo 2: a função abaixo já trata os dois casos juntos.
#
# ── SE QUISER OTIMIZAR O NÍVEL DO STOP ─────────────────────────
#
#   Adicione diferentes valores de stop na grade para que o
#   walk-forward encontre o melhor nível automaticamente:
#
#     for short in range(3, 16):
#         for long_ in range(30, 71):
#             for stop in [0.03, 0.05, 0.08, 0.10]:
#                 grade.append({
#                     'short'     : short,
#                     'long'      : long_,
#                     'stop_loss' : stop
#                 })
#
#   Isso testa 4 níveis de stop para cada combinação de MAs.
# ────────────────────────────────────────────────────────────────

def aplicar_stop(strategy, janela, params):
    """
    Aplica stop loss e/ou stop gain na série de retornos.

    Não edite esta função — apenas adicione 'stop_loss' e/ou
    'stop_gain' no dicionário da sua grade (Seção C) se quiser
    usar stops. Se não adicionar, ela retorna sem alterar nada.

    Como funciona internamente:
      - Percorre os retornos diários um a um
      - Acumula o retorno desde a última entrada no mercado
      - Se a perda acumulada atingir stop_loss → zera o retorno
        e reinicia o contador (simula saída e nova entrada)
      - Se o ganho acumulado atingir stop_gain → mesma coisa
    """

    stop_loss = params.get('stop_loss', None)
    stop_gain = params.get('stop_gain', None)

    # Se não há nenhum stop definido na grade, retorna sem alteração
    if stop_loss is None and stop_gain is None:
        return strategy

    strategy = strategy.copy()
    retorno_acumulado_trade = 0.0

    for i in range(len(strategy)):

        # Atualiza o retorno acumulado desde a entrada
        retorno_acumulado_trade = (
            (1 + retorno_acumulado_trade) *
            (1 + float(strategy.iloc[i])) - 1
        )

        # Verifica stop loss: perdemos mais que o limite?
        stop_loss_atingido = (
            stop_loss is not None and
            retorno_acumulado_trade < -stop_loss
        )

        # Verifica stop gain: ganhamos mais que o limite?
        stop_gain_atingido = (
            stop_gain is not None and
            retorno_acumulado_trade > stop_gain
        )

        if stop_loss_atingido or stop_gain_atingido:
            # Zera o retorno deste dia (saída forçada)
            # e reinicia o contador para o próximo trade
            strategy.iloc[i]        = 0.0
            retorno_acumulado_trade  = 0.0

    return strategy

## CÉLULA 3 — 🔒 Parâmetros do walk-forward

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
# Define as janelas de treino e teste do walk-forward.
#
# train_years = quantos anos usar para encontrar os melhores parâmetros
# test_years  = quantos anos usar para testar fora da amostra (sempre 1)

train_years = 5
test_years  = 1

## CÉLULA 4 — 🔒 Função de custo de trade

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
#
# Esta função desconta o custo de corretagem da estratégia.
# É chamada automaticamente pelo walk-forward.
#
# Como funciona:
#   - Detecta os dias de ENTRADA (sinal 0→1) e SAÍDA (sinal 1→0)
#   - Subtrai o custo percentual apenas nesses dias
#   - Dias normais (sem mudança de posição) não têm custo

def _aplicar_custo(signal, strategy, custo):

    # Detecta mudanças de posição: diff() calcula
    # a diferença entre o sinal de hoje e o de ontem.
    # abs() trata entrada e saída da mesma forma.
    sinal_deslocado = signal.shift(1)
    mudanca         = sinal_deslocado.diff().abs()
    custos          = mudanca.fillna(0) * custo

    # Alinha os índices antes de subtrair
    custos = custos.reindex(strategy.index).fillna(0)

    return strategy - custos

## CÉLULA 5 — 🔒 Função principal do walk-forward

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
#
# Esta função executa o loop completo do walk-forward.
# Ela usa as funções que seu grupo define na célula do setup:
#
#   gerar_grade()            → quais combinações de parâmetros testar
#   calcular_sinal()         → a lógica de entrada/saída da estratégia
#   aplicar_stop() (opcional)→ stop loss / take profit
#
# Você não precisa entender o código aqui —
# só precisa preencher corretamente a célula do setup.

def rodar_walk_forward(data, years, train_years, test_years,
                       gerar_grade, calcular_sinal,
                       aplicar_stop=None, custo_trade=0.003):
    """
    Parâmetros
    ----------
    data         : DataFrame com os preços
    years        : lista de anos disponíveis
    train_years  : anos de treino por janela
    test_years   : anos de teste por janela
    gerar_grade  : função que retorna lista de dicts de parâmetros
    calcular_sinal : função que recebe (janela, params) e retorna sinal
    aplicar_stop : função opcional de stop loss
    custo_trade  : custo por operação em decimal (ex: 0.003 = 0.3%)

    Retorna
    -------
    records : lista de dicionários com os resultados de cada janela
    """

    records = []
    grade   = gerar_grade()

    for i in range(len(years) - train_years - test_years):

        # ---- Definir janelas ----
        train_start = years[i]
        train_end   = years[i + train_years - 1]
        test_year   = years[i + train_years]

        train = data[
            (data['Year'] >= train_start) &
            (data['Year'] <= train_end)
        ].copy()

        test = data[data['Year'] == test_year].copy()

        # ---- Busca pelo melhor parâmetro no treino ----
        best_sharpe = -999
        best_params = None

        for params in grade:

            signal  = calcular_sinal(train, params)
            returns = train['Close'].pct_change()

            strategy = signal.shift(1) * returns

            if aplicar_stop is not None:
                strategy = aplicar_stop(strategy, train, params)

            strategy = _aplicar_custo(signal, strategy, custo_trade)
            strategy = strategy.dropna()

            std = float(strategy.std())
            if std == 0 or np.isnan(std):
                continue

            sharpe = float(strategy.mean()) / std

            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_params = params

        if best_params is None:
            continue

        # ---- Aplicar melhor parâmetro no teste ----
        signal  = calcular_sinal(test, best_params)
        returns = test['Close'].pct_change()

        strategy = signal.shift(1) * returns

        if aplicar_stop is not None:
            strategy = aplicar_stop(strategy, test, best_params)

        strategy = _aplicar_custo(signal, strategy, custo_trade)
        strategy = strategy.dropna()

        retorno_oos = float((1 + strategy).prod() - 1)

        # ---- Guardar resultado ----
        record = {
            'ano_teste'        : test_year,
            'retorno_oos'      : retorno_oos,
            'sharpe_treino'    : best_sharpe,
            'retornos_diarios' : strategy,
        }
        record.update(best_params)
        records.append(record)

    print(f'Walk-forward concluído: {len(records)} janelas de teste.')
    return records

## CÉLULA 6 — 🔒 Função de métricas

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
# Calcula todas as métricas do backtest a partir dos records.

def calcular_metricas(records):

    DIAS_ANO = 252
    df = pd.DataFrame(records)
    r  = df['retorno_oos'].astype(float)
    n  = len(r)

    # Performance
    retorno_acumulado   = float((1 + r).prod() - 1)
    cagr                = float((1 + retorno_acumulado) ** (1 / n) - 1)
    retorno_medio_anual = float(r.mean())

    # Risco
    volatilidade = float(r.std())
    equity_curve = (1 + r).cumprod()
    peak         = equity_curve.cummax()
    drawdown     = (equity_curve / peak) - 1
    max_drawdown = float(drawdown.min())
    ulcer_index  = float(np.sqrt((drawdown ** 2).mean()))

    # Risco-retorno
    std_r  = float(r.std())
    sharpe = float(r.mean()) / std_r if std_r != 0 else np.nan

    anos_negativos = r[r < 0]
    if len(anos_negativos) >= 2:
        downside_std = float(anos_negativos.std())
        sortino = float(r.mean()) / downside_std if downside_std != 0 else np.nan
    else:
        sortino = np.nan

    calmar = cagr / abs(max_drawdown) if abs(max_drawdown) > 0.0001 else np.nan

    # Operacional (requer retornos_diarios)
    if 'retornos_diarios' in df.columns:
        todos = pd.concat(
            df['retornos_diarios'].tolist()
        ).reset_index(drop=True).astype(float)

        posicionado   = (todos != 0).astype(int)
        dias_comprado = todos[posicionado == 1]
        num_trades    = int((posicionado.diff() == 1).sum())
        win_rate      = float((dias_comprado > 0).mean())
        ganhos        = float(dias_comprado[dias_comprado > 0].sum())
        perdas        = float(abs(dias_comprado[dias_comprado < 0].sum()))
        profit_factor = ganhos / perdas if perdas > 0 else np.nan
        exposicao     = float(posicionado.mean())

        std_d = float(todos.std())
        sharpe_diario = float(
            todos.mean() / std_d * np.sqrt(DIAS_ANO)
        ) if std_d != 0 else np.nan

        equity_diaria   = (1 + todos).cumprod()
        peak_d          = equity_diaria.cummax()
        drawdown_diario = (equity_diaria / peak_d) - 1
        max_dd_diario   = float(drawdown_diario.min())

    else:
        todos = num_trades = win_rate = profit_factor = None
        exposicao = sharpe_diario = equity_diaria = np.nan
        drawdown_diario = max_dd_diario = None

    # Distribuição
    skewness = float(r.skew())
    kurtosis = float(r.kurt())

    # Consistência
    pct_positivos  = float((r > 0).mean())
    sharpe_por_ano = (r / std_r).tolist() if std_r != 0 else [np.nan] * n

    return dict(
        df=df, r=r, n_anos=n,
        retorno_acumulado=retorno_acumulado, cagr=cagr,
        retorno_medio_anual=retorno_medio_anual,
        volatilidade=volatilidade, equity_curve=equity_curve,
        drawdown=drawdown, max_drawdown=max_drawdown,
        ulcer_index=ulcer_index, sharpe=sharpe,
        sharpe_diario=sharpe_diario, sortino=sortino, calmar=calmar,
        num_trades=num_trades, win_rate=win_rate,
        profit_factor=profit_factor, exposicao=exposicao,
        todos_diarios=todos, equity_diaria=equity_diaria,
        drawdown_diario=drawdown_diario, max_dd_diario=max_dd_diario,
        skewness=skewness, kurtosis=kurtosis,
        pct_positivos=pct_positivos, sharpe_por_ano=sharpe_por_ano,
    )


def imprimir_metricas(m):

    print('=' * 52)
    print('  RESUMO DE MÉTRICAS — WALK-FORWARD OOS')
    print('=' * 52)

    print('\n--- PERFORMANCE ---')
    print(f"  Retorno acumulado      : {m['retorno_acumulado']:.2%}")
    print(f"  CAGR                   : {m['cagr']:.2%}")
    print(f"  Retorno médio anual    : {m['retorno_medio_anual']:.2%}")

    print('\n--- RISCO ---')
    print(f"  Volatilidade anual     : {m['volatilidade']:.2%}")
    md = m['max_drawdown']
    print(f"  Máximo Drawdown        : {md:.2%}" +
          ("  ← todos os anos positivos" if md == 0 else ""))
    ui = m['ulcer_index']
    print(f"  Ulcer Index            : {ui:.4f}" +
          ("  ← sem períodos de queda" if ui == 0 else ""))

    print('\n--- RISCO-RETORNO ---')
    print(f"  Sharpe (anual)         : {m['sharpe']:.3f}")
    if not np.isnan(m['sharpe_diario']):
        print(f"  Sharpe (diário ×√252)  : {m['sharpe_diario']:.3f}")
    n_neg = int((m['r'] < 0).sum())
    print(f"  Sortino Ratio          : " +
          (f"{m['sortino']:.3f}" if not np.isnan(m['sortino'])
           else f"n/a  ← só {n_neg} ano(s) negativo(s)"))
    print(f"  Calmar Ratio           : " +
          (f"{m['calmar']:.3f}" if not np.isnan(m['calmar'])
           else "n/a  ← drawdown = 0"))

    print('\n--- OPERACIONAL ---')
    print(f"  Nº de trades           : {m['num_trades']}")
    if m['win_rate'] is not None and not np.isnan(m['win_rate']):
        print(f"  Win Rate (diário)      : {m['win_rate']:.2%}")
        print(f"  Profit Factor          : {m['profit_factor']:.3f}")
        print(f"  Exposição ao mercado   : {m['exposicao']:.2%}")

    print('\n--- DISTRIBUIÇÃO ---')
    print(f"  Skewness               : {m['skewness']:.3f}")
    print(f"  Kurtosis (excesso)     : {m['kurtosis']:.3f}")

    print('\n--- CONSISTÊNCIA ---')
    print(f"  Anos positivos         : {int((m['r'] > 0).sum())} "
          f"de {m['n_anos']}  ({m['pct_positivos']:.0%})")

## CÉLULA 7 — 🔒 Função de gráficos do Walk-Forward

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
# Gera os 9 gráficos de análise do walk-forward.

def plotar_resultados(m, titulo='Walk-Forward — Análise Completa OOS'):

    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    fig.suptitle(titulo, fontsize=14)
    ax  = axes.flatten()
    pct = plt.FuncFormatter(lambda x, _: f'{x:.0%}')

    df             = m['df']
    r              = m['r']
    equity_curve   = m['equity_curve']
    drawdown       = m['drawdown']
    sharpe_por_ano = m['sharpe_por_ano']

    # -- Linha 1: desempenho geral --

    ax[0].plot(df['ano_teste'], equity_curve, marker='o', color='steelblue')
    ax[0].axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
    ax[0].set_title('Curva de Capital (OOS anual)')
    ax[0].set_xlabel('Ano')
    ax[0].set_ylabel('Capital (início = R$1)')

    cores = ['green' if v >= 0 else 'red' for v in r]
    ax[1].bar(df['ano_teste'], r, color=cores, edgecolor='white')
    ax[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax[1].set_title('Retorno Anual por Janela (OOS)')
    ax[1].set_xlabel('Ano')
    ax[1].yaxis.set_major_formatter(pct)

    ax[2].fill_between(df['ano_teste'], drawdown, 0,
                       color='red', alpha=0.4)
    ax[2].plot(df['ano_teste'], drawdown, color='red', linewidth=1)
    ax[2].set_title(f"Drawdown Anual (máx: {m['max_drawdown']:.1%})")
    ax[2].set_xlabel('Ano')
    ax[2].yaxis.set_major_formatter(pct)

    # -- Linha 2: parâmetros e consistência --

    # Detecta automaticamente as colunas de parâmetros do grupo
    colunas_fixas  = {'ano_teste','retorno_oos','sharpe_treino',
                      'retornos_diarios','acumulado'}
    colunas_params = [c for c in df.columns if c not in colunas_fixas]
    cores_p  = ['darkorange','steelblue','green','purple','brown']
    markers  = ['s', '^', 'o', 'D', 'v']

    for idx, col in enumerate(colunas_params):
        ax[3].plot(df['ano_teste'], df[col],
                   marker=markers[idx % len(markers)],
                   color=cores_p[idx % len(cores_p)],
                   label=col)
    ax[3].set_title('Parâmetros Selecionados por Janela')
    ax[3].set_xlabel('Ano')
    ax[3].set_ylabel('Valor')
    if colunas_params:
        ax[3].legend(fontsize=8)

    cores_s = ['green' if v >= 0 else 'red' for v in sharpe_por_ano]
    ax[4].bar(df['ano_teste'], sharpe_por_ano,
              color=cores_s, edgecolor='white')
    ax[4].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax[4].set_title('Sharpe por Ano (OOS)')
    ax[4].set_xlabel('Ano')

    ax[5].hist(r, bins=8, edgecolor='white', color='steelblue')
    ax[5].axvline(float(r.mean()), color='red', linestyle='--',
                  linewidth=1.5, label=f'Média: {float(r.mean()):.1%}')
    ax[5].axvline(0, color='orange', linestyle='--', linewidth=1.0)
    ax[5].set_title('Distribuição Retornos Anuais')
    ax[5].xaxis.set_major_formatter(pct)
    ax[5].legend(fontsize=8)

    # -- Linha 3: análise diária --

    if m['equity_diaria'] is not None:
        eq_d = m['equity_diaria']
        dd_d = m['drawdown_diario']
        ret_comprado = m['todos_diarios']
        ret_comprado = ret_comprado[ret_comprado != 0]

        ax[6].plot(eq_d.values, linewidth=0.7, color='steelblue')
        ax[6].axhline(1.0, color='gray', linestyle='--', linewidth=0.6)
        ax[6].set_title('Curva de Capital Diária (OOS)')
        ax[6].set_xlabel('Dias de pregão')

        ax[7].hist(ret_comprado, bins=40, edgecolor='white',
                   color='steelblue', alpha=0.8)
        ax[7].axvline(float(ret_comprado.mean()), color='red',
                      linestyle='--', linewidth=1.5,
                      label=f'Média: {float(ret_comprado.mean()):.2%}')
        ax[7].axvline(0, color='orange', linestyle='--', linewidth=1.0)
        ax[7].set_title('Distribuição Retornos Diários')
        ax[7].xaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
        ax[7].legend(fontsize=8)

        ax[8].fill_between(range(len(dd_d)), dd_d.values, 0,
                           color='red', alpha=0.4)
        ax[8].plot(dd_d.values, color='red', linewidth=0.6)
        ax[8].set_title(f"Drawdown Diário (máx: {m['max_dd_diario']:.1%})")
        ax[8].set_xlabel('Dias de pregão')
        ax[8].yaxis.set_major_formatter(pct)

    else:
        for idx, titulo_g in enumerate(
                ['Curva Diária','Retornos Diários','Drawdown Diário']):
            ax[6+idx].text(0.5, 0.5, 'Dados diários\nnão disponíveis.',
                           ha='center', va='center', fontsize=10,
                           transform=ax[6+idx].transAxes, color='gray')
            ax[6+idx].set_title(titulo_g)

    plt.tight_layout()
    plt.show()

## CÉLULA 8 — 🔒 Funções dos testes estatísticos e de robustez

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
#
# Este módulo implementa 5 testes de validação estatística:
#
#   1. t-Student      → o retorno médio é significativamente positivo?
#   2. Bootstrap      → intervalo de confiança do Sharpe por reamostagem
#   3. Monte Carlo    → distribuição de resultados possíveis
#   4. Deflated Sharpe→ penaliza o Sharpe pelo número de estratégias testadas
#   5. White's Reality Check → corrige o viés de data-snooping
#
# Todos recebem o dicionário de métricas 'm' produzido por calcular_metricas()
# e retornam um dicionário com os resultados do teste.

from scipy import stats as scipy_stats

N_SIMULACOES = 10_000   # número de simulações para Bootstrap e Monte Carlo


# ======================================================
# TESTE 1 — t-Student
# ======================================================
#
# Pergunta: o retorno médio OOS é estatisticamente diferente de zero?
# Ou seja: a estratégia tem edge real, ou poderia ser resultado do acaso?
#
# Como funciona:
#   Aplicamos o teste t de uma amostra (one-sample t-test):
#   H0 (hipótese nula):       retorno médio = 0  (sem edge)
#   H1 (hipótese alternativa): retorno médio > 0  (com edge)
#
#   Se o p-valor for menor que 0.05, rejeitamos H0 com 95% de confiança.
#
# Limitação importante:
#   Com ~9 janelas OOS, a amostra é pequena.
#   O teste t assume distribuição normal dos retornos.
#   Um p-valor alto aqui NÃO significa que a estratégia é ruim —
#   pode apenas significar que não temos dados suficientes.

def teste_t_student(m):
    """
    Recebe o dicionário de métricas e retorna o resultado do teste t.
    """
    r = m['r'].values  # array numpy com os retornos anuais OOS

    # ttest_1samp testa se a média de 'r' é significativamente
    # diferente de 'popmean' (aqui, zero).
    # alternative='greater' testa se é MAIOR que zero (teste unilateral).
    t_stat, p_valor_bilateral = scipy_stats.ttest_1samp(r, popmean=0)

    # Convertemos para teste unilateral (só nos interessa se > 0)
    # p unilateral = p bilateral / 2 (quando t_stat > 0)
    p_valor = p_valor_bilateral / 2 if t_stat > 0 else 1.0

    # Graus de liberdade = n - 1
    graus_liberdade = len(r) - 1

    # Intervalo de confiança de 95% para a média
    ic_95 = scipy_stats.t.interval(
        confidence   = 0.95,
        df           = graus_liberdade,
        loc          = r.mean(),
        scale        = scipy_stats.sem(r)  # erro padrão da média
    )

    significativo = p_valor < 0.05

    return {
        't_stat'          : float(t_stat),
        'p_valor'         : float(p_valor),
        'graus_liberdade' : graus_liberdade,
        'ic_95_low'       : float(ic_95[0]),
        'ic_95_high'      : float(ic_95[1]),
        'significativo'   : significativo,
        'conclusao'       : (
            'Retorno significativamente positivo (p < 0.05)'
            if significativo else
            'Não é possível rejeitar H0 (p ≥ 0.05) — amostra pequena'
        )
    }


# ======================================================
# TESTE 2 — Bootstrap do Sharpe
# ======================================================
#
# Pergunta: qual o intervalo de confiança real do Sharpe?
#
# O problema: com ~9 observações, o Sharpe calculado é
# uma estimativa ruidosa. O Bootstrap quantifica essa incerteza.
#
# Como funciona:
#   1) Reamostrar os retornos OOS com reposição N vezes
#   2) Calcular o Sharpe de cada reamostagem
#   3) A distribuição desses Sharpes revela o intervalo de confiança
#
# Reamostrar "com reposição" significa sortear aleatoriamente
# os retornos, podendo repetir o mesmo ano mais de uma vez.
# Isso simula "outras possíveis histórias" com os mesmos dados.

def bootstrap_sharpe(m, n_sim=N_SIMULACOES, seed=42):
    """
    Calcula o intervalo de confiança do Sharpe por Bootstrap.
    """
    np.random.seed(seed)  # garante reprodutibilidade dos resultados
    r = m['r'].values
    n = len(r)

    sharpes_bootstrap = []

    for _ in range(n_sim):
        # Reamostrar com reposição: sorteia 'n' índices aleatórios,
        # podendo repetir. Simula uma "história alternativa" dos retornos.
        amostra = np.random.choice(r, size=n, replace=True)

        std_amostra = amostra.std()
        if std_amostra != 0:
            sharpes_bootstrap.append(amostra.mean() / std_amostra)

    sharpes_bootstrap = np.array(sharpes_bootstrap)

    return {
        'sharpes_simulados' : sharpes_bootstrap,
        'sharpe_original'   : m['sharpe'],
        'ic_95_low'         : float(np.percentile(sharpes_bootstrap, 2.5)),
        'ic_95_high'        : float(np.percentile(sharpes_bootstrap, 97.5)),
        'ic_99_low'         : float(np.percentile(sharpes_bootstrap, 0.5)),
        'ic_99_high'        : float(np.percentile(sharpes_bootstrap, 99.5)),
        'prob_sharpe_pos'   : float((sharpes_bootstrap > 0).mean()),
        'conclusao'         : (
            f'Sharpe original: {m["sharpe"]:.3f} | '
            f'IC 95%: [{np.percentile(sharpes_bootstrap, 2.5):.3f}, '
            f'{np.percentile(sharpes_bootstrap, 97.5):.3f}]'
        )
    }


# ======================================================
# TESTE 3 — Monte Carlo
# ======================================================
#
# Pergunta: se sortearmos aleatoriamente os retornos diários,
# qual seria a distribuição de resultados possíveis?
#
# Como funciona:
#   1) Tomar os retornos diários OOS reais
#   2) Embaralhar aleatoriamente a ordem desses retornos N vezes
#   3) Calcular o retorno acumulado e o drawdown de cada embaralhamento
#   4) Ver onde o resultado real se posiciona nessa distribuição
#
# Diferença do Bootstrap:
#   - Bootstrap: reamostrar COM reposição (pode repetir retornos)
#   - Monte Carlo aqui: embaralhar SEM reposição (preserva os retornos,
#     muda apenas a ordem — testa se a SEQUÊNCIA importa)
#
# Se o resultado real for melhor que 95% dos embaralhamentos,
# a ordem dos retornos (timing) agrega valor à estratégia.

def monte_carlo(m, n_sim=N_SIMULACOES, seed=42):
    """
    Simula distribuições de retorno acumulado e drawdown
    embaralhando os retornos diários aleatoriamente.
    """

    if m['todos_diarios'] is None:
        return {'erro': 'Retornos diários não disponíveis.'}

    np.random.seed(seed)
    retornos = m['todos_diarios'].values

    retornos_acumulados_sim = []
    max_drawdowns_sim       = []

    for _ in range(n_sim):
        # Embaralha a ordem dos retornos (sem reposição)
        np.random.shuffle(retornos)

        # Calcula equity curve desta simulação
        eq   = (1 + retornos).cumprod()
        peak = np.maximum.accumulate(eq)
        dd   = (eq / peak) - 1

        retornos_acumulados_sim.append(eq[-1] - 1)
        max_drawdowns_sim.append(dd.min())

    ret_sim = np.array(retornos_acumulados_sim)
    dd_sim  = np.array(max_drawdowns_sim)

    # Resultado real para comparação
    ret_real = m['retorno_acumulado']
    dd_real  = m['max_dd_diario'] if m['max_dd_diario'] is not None else np.nan

    # Percentil do resultado real na distribuição simulada
    percentil_retorno  = float((ret_sim < ret_real).mean())
    percentil_drawdown = float((dd_sim  > dd_real ).mean())  # drawdown é negativo

    return {
        'retornos_simulados'  : ret_sim,
        'drawdowns_simulados' : dd_sim,
        'retorno_real'        : ret_real,
        'drawdown_real'       : dd_real,
        'percentil_retorno'   : percentil_retorno,
        'percentil_drawdown'  : percentil_drawdown,
        'conclusao_retorno'   : (
            f'Retorno real supera {percentil_retorno:.0%} das simulações aleatórias'
        ),
        'conclusao_drawdown'  : (
            f'Drawdown real é melhor que {percentil_drawdown:.0%} das simulações'
        )
    }


# ======================================================
# TESTE 4 — Deflated Sharpe Ratio (DSR)
# ======================================================
#
# Pergunta: o Sharpe encontrado é genuíno ou é fruto de
# termos testado muitas combinações de parâmetros?
#
# O problema (data-snooping / multiple testing):
#   Se testarmos 533 combinações de parâmetros e pegarmos
#   a melhor, o Sharpe observado estará inflado pelo acaso.
#   É como jogar um dado 533 vezes e dizer "olha, tirei 6!"
#
# O DSR corrige isso calculando o Sharpe mínimo esperado
# pelo acaso dado o número de estratégias testadas (n_testes),
# e verifica se o Sharpe real supera esse limiar com
# significância estatística.
#
# Referência: Bailey & López de Prado (2014)
# "The Deflated Sharpe Ratio: Correcting for Selection Bias,
#  Backtest Overfitting and Non-Normality"

def deflated_sharpe(m, n_testes, n_anos_treino=None):
    """
    Calcula o Deflated Sharpe Ratio.

    Parâmetros
    ----------
    m            : dicionário de métricas
    n_testes     : número de combinações de parâmetros testadas
                   (use len(gerar_grade()) para obter este valor)
    n_anos_treino: total de anos no período de treino (opcional,
                   melhora a estimativa do Sharpe esperado pelo acaso)

    Retorna
    -------
    dicionário com o DSR e a conclusão
    """

    sharpe_obs = m['sharpe']   # Sharpe observado (anual, não anualizado)
    n          = m['n_anos']   # número de janelas OOS
    sk         = m['skewness']
    ku         = m['kurtosis'] # kurtosis de excesso

    # --------------------------------------------------
    # Passo 1: Sharpe esperado pelo acaso
    # --------------------------------------------------
    #
    # Quando testamos muitas combinações, esperamos que a melhor
    # tenha um Sharpe positivo só por sorte.
    # Essa fórmula (Bailey & López de Prado) estima esse valor esperado:
    #
    #   E[max Sharpe] ≈ Z^{-1}(1 - 1/n_testes) × sqrt(variância_sharpe)
    #
    # Onde Z^{-1} é o quantil da normal padrão.
    # Usamos a aproximação pelo valor esperado do máximo de
    # n_testes normais padrão.

    # Valor esperado do máximo de n_testes normais padrão
    # Approximação: E[max(Z1...Zn)] ≈ (1 - γ) × Z^{-1}(1-1/n) + γ × Z^{-1}(1-1/(n×e))
    # Simplificamos para a forma mais comum da literatura:
    euler_gamma = 0.5772  # constante de Euler-Mascheroni
    e_max_z = (
        (1 - euler_gamma) * scipy_stats.norm.ppf(1 - 1 / n_testes) +
        euler_gamma       * scipy_stats.norm.ppf(1 - 1 / (n_testes * np.e))
    )

    # O Sharpe esperado pelo acaso escala com a raiz do número de observações
    sharpe_esperado = e_max_z / np.sqrt(n)

    # --------------------------------------------------
    # Passo 2: variância do Sharpe estimado
    # --------------------------------------------------
    #
    # O Sharpe calculado com poucos dados tem muita incerteza.
    # Esta fórmula (Mertens 2002) ajusta pela assimetria (skewness)
    # e curtose (kurtosis) dos retornos — distribuições não-normais
    # inflam a variância do Sharpe estimado.
    #
    # Termos da fórmula:
    #   1/n               → incerteza básica com n observações
    #   sk * sharpe_obs   → penalidade pela assimetria
    #   (ku/4) * sharpe²  → penalidade pela curtose

    var_sharpe = (
        (1 / n) *
        (1 + (sk * sharpe_obs) + ((ku / 4) * sharpe_obs ** 2))
    )
    std_sharpe = np.sqrt(max(var_sharpe, 1e-10))  # evita raiz de negativo

    # --------------------------------------------------
    # Passo 3: calcular o DSR
    # --------------------------------------------------
    #
    # DSR = probabilidade de que o Sharpe real supere
    # o Sharpe esperado pelo acaso.
    #
    # Fórmula: Φ( (SR_obs - SR_esperado) / std(SR) )
    # Onde Φ é a função de distribuição acumulada normal.

    dsr = float(scipy_stats.norm.cdf(
        (sharpe_obs - sharpe_esperado) / std_sharpe
    ))

    return {
        'sharpe_observado' : sharpe_obs,
        'sharpe_esperado'  : float(sharpe_esperado),
        'dsr'              : dsr,
        'n_testes'         : n_testes,
        'aprovado'         : dsr > 0.95,
        'conclusao'        : (
            f'DSR = {dsr:.3f} | '
            f'Sharpe obs: {sharpe_obs:.3f} | '
            f'Sharpe esp. pelo acaso: {sharpe_esperado:.3f} | ' +
            ('✓ Sharpe genuíno (DSR > 0.95)'
             if dsr > 0.95 else
             '✗ Sharpe possivelmente inflado pelo número de testes')
        )
    }

# ======================================================
# TESTE 5 — White's Reality Check (versão simplificada)
# ======================================================
#
# Pergunta: a melhor estratégia da grade realmente supera
# o benchmark (buy-and-hold), ou foi selecionada por sorte
# após testarmos centenas de combinações?
#
# Versão implementada: Reality Check de White (2000)
# aplicado à grade de parâmetros do próprio grupo.
#
# Como funciona:
#   1) Calculamos o retorno de TODAS as combinações da grade
#      no período OOS completo
#   2) Comparamos cada uma com o benchmark (buy-and-hold)
#   3) Usamos Bootstrap para construir a distribuição nula:
#      "qual seria o melhor desempenho esperado por acaso?"
#   4) Se a melhor estratégia superar o quantil 95% da
#      distribuição nula, rejeitamos H0
#
# H0 (hipótese nula): nenhuma estratégia da grade supera
#                     o benchmark de forma genuína

def whites_reality_check(data, years, train_years, test_years,
                         gerar_grade, calcular_sinal,
                         custo_trade=0.003,
                         n_sim=N_SIMULACOES, seed=42):
    """
    Executa o White's Reality Check com Bootstrap vetorizado.

    A diferença principal em relação à versão anterior é que
    o Bootstrap é feito de forma VETORIZADA usando numpy:
    em vez de um loop Python para cada simulação, geramos
    todas as 10.000 reamostagens de uma vez em matriz.

    Isso reduz o tempo de 10k simulações de ~60s para ~3s.

    Parâmetros
    ----------
    data, years, train_years, test_years : mesmos do walk-forward
    gerar_grade, calcular_sinal          : funções do setup do grupo
    custo_trade                          : custo por operação
    n_sim                                : simulações Bootstrap (10.000)
    seed                                 : semente aleatória

    Retorna
    -------
    dicionário com o resultado do teste
    """

    np.random.seed(seed)
    grade = gerar_grade()

    # --------------------------------------------------
    # Passo 1: calcular retornos diários OOS de CADA combinação
    # --------------------------------------------------
    #
    # Para cada conjunto de parâmetros, calculamos o retorno
    # diário em TODOS os períodos OOS sem otimização —
    # os mesmos parâmetros fixos em todo o período.
    #
    # Guardamos em uma LISTA DE ARRAYS numpy (um por estratégia).
    # Isso permite vetorizar o Bootstrap no passo 4.

    print(f'Calculando {len(grade)} combinações da grade... ', end='')

    # Primeiro, descobrimos o tamanho total da série OOS
    # para poder pré-alocar a matriz de retornos.
    n_janelas = len(years) - train_years - test_years

    # Calculamos os retornos do benchmark (buy-and-hold) uma única vez.
    # Reutilizamos em todas as estratégias.
    retornos_bh_lista = []
    for i in range(n_janelas):
        test_year = years[i + train_years]
        test      = data[data['Year'] == test_year].copy()
        bh        = test['Close'].pct_change().dropna().values
        retornos_bh_lista.append(bh)

    bh_array = np.concatenate(retornos_bh_lista)  # série contínua do benchmark
    n_obs    = len(bh_array)

    # Matriz de excessos: shape (n_estrategias, n_obs)
    # Cada linha = retornos diários de uma estratégia MENOS o benchmark.
    # Valores positivos = estratégia supera o benchmark naquele dia.
    excesso_matrix = np.zeros((len(grade), n_obs))

    for k, params in enumerate(grade):

        retornos_lista = []

        for i in range(n_janelas):
            test_year = years[i + train_years]
            test      = data[data['Year'] == test_year].copy()

            signal   = calcular_sinal(test, params)
            returns  = test['Close'].pct_change()
            strategy = signal.shift(1) * returns
            strategy = _aplicar_custo(signal, strategy, custo_trade)
            strategy = strategy.dropna().values

            retornos_lista.append(strategy)

        ret_array = np.concatenate(retornos_lista)

        # Alinha tamanhos caso haja diferença de 1-2 dias por NaN
        n_min = min(len(ret_array), n_obs)
        excesso_matrix[k, :n_min] = ret_array[:n_min] - bh_array[:n_min]

    print('OK')

    # --------------------------------------------------
    # Passo 2: média de excesso de cada estratégia
    # --------------------------------------------------
    #
    # Para cada estratégia, calculamos a média diária de
    # excesso sobre o benchmark.
    # A melhor estratégia é a de maior média.

    media_excesso = excesso_matrix.mean(axis=1)   # shape: (n_estrategias,)
    idx_melhor    = np.argmax(media_excesso)
    melhor_excesso = media_excesso[idx_melhor]
    melhor_params  = grade[idx_melhor]


    # --------------------------------------------------
    # Passo 3: Bootstrap VETORIZADO
    # --------------------------------------------------
    #
    # A ideia é a mesma do Bootstrap anterior:
    # reamostrar os dias com reposição e ver qual seria
    # o melhor desempenho esperado por acaso.
    #
    # Versão vetorizada:
    #   Em vez de um loop Python (lento), geramos de uma vez
    #   uma MATRIZ de índices aleatórios de shape (n_sim, n_obs).
    #   Depois usamos indexação numpy para calcular todas as
    #   10.000 reamostagens simultaneamente.
    #
    # Por que isso é muito mais rápido?
    #   Loop Python:  10.000 iterações × overhead Python por iteração
    #   Numpy:        1 operação em C sobre uma matriz inteira
    #   Speedup típico: 20-50x mais rápido.

    print(f'Rodando {n_sim:,} simulações Bootstrap vetorizadas... ', end='')

    # Gera todos os índices de reamostagem de uma vez:
    # shape (n_sim, n_obs) — cada linha é uma reamostagem
    idx_boot = np.random.randint(0, n_obs, size=(n_sim, n_obs))

    # Para cada simulação, calculamos a média de excesso
    # de TODAS as estratégias ao mesmo tempo.
    #
    # excesso_matrix[:, idx_boot] teria shape (n_estrategias, n_sim, n_obs)
    # — muito grande para a memória do Colab.
    #
    # Solução: processamos em lotes de 500 simulações por vez.
    # Isso mantém o uso de memória controlado (~200MB máximo)
    # sem sacrificar a velocidade.

    LOTE = 500   # simulações por lote — ajuste se o Colab der MemoryError
    maximos_bootstrap = np.zeros(n_sim)

    for inicio in range(0, n_sim, LOTE):
        fim        = min(inicio + LOTE, n_sim)
        idx_lote   = idx_boot[inicio:fim]          # shape: (LOTE, n_obs)

        # Para cada lote, calculamos:
        #   excesso_matrix[:, idx_lote]  → shape: (n_estrategias, LOTE, n_obs)
        #   .mean(axis=2)                → shape: (n_estrategias, LOTE)
        #   .max(axis=0)                 → shape: (LOTE,) — máximo entre estratégias

        exc_lote = excesso_matrix[:, idx_lote].mean(axis=2)  # (n_estrat, LOTE)
        maximos_bootstrap[inicio:fim] = exc_lote.max(axis=0) # (LOTE,)

    print('OK')


    # --------------------------------------------------
    # Passo 4: calcular p-valor
    # --------------------------------------------------
    #
    # p-valor = fração das 10.000 simulações onde o máximo
    # aleatório superou o melhor desempenho real observado.
    #
    # Pequeno p-valor → difícil de explicar pelo acaso → rejeita H0.
    # Grande p-valor → resultado pode ser fruto de data-snooping.

    p_valor  = float((maximos_bootstrap >= melhor_excesso).mean())
    aprovado = p_valor < 0.05

    return {
        'melhor_params'     : melhor_params,
        'melhor_excesso'    : float(melhor_excesso),
        'p_valor'           : p_valor,
        'maximos_bootstrap' : maximos_bootstrap,
        'aprovado'          : aprovado,
        'n_estrategias'     : len(grade),
        'conclusao'         : (
            f'p-valor = {p_valor:.3f} | '
            f'Melhor excesso s/ benchmark: {melhor_excesso:.5f}/dia | '
            f'{len(grade)} estratégias testadas | ' +
            ('✓ Rejeita H0: melhor estratégia supera benchmark genuinamente'
             if aprovado else
             '✗ Não rejeita H0: resultado pode ser fruto de data-snooping')
        )
    }

**O que mudou e por quê:**

A única mudança visível para os alunos é `N_SIMULACOES = 10_000`. Internamente, a grande mudança é a vetorização do Bootstrap no White's Reality Check:
```
Versão anterior (loop Python):
  for _ in range(10_000):           # 10.000 iterações Python
      idx = random.choice(...)      # 1 reamostagem por vez
      exc = excesso[:, idx].mean()  # 1 cálculo por vez
  → ~60 segundos

Versão nova (numpy vetorizado em lotes):
  idx = randint(shape=(10_000, n_obs))  # todos os índices de uma vez
  for lote in range(0, 10_000, 500):    # 20 lotes de 500
      exc = excesso[:, lote].mean()     # 500 cálculos simultâneos em C
  → ~5-8 segundos

  O processamento em lotes de 500 é o detalhe mais importante: sem ele, a matriz excesso_matrix[:, idx_boot] com 533 estratégias × 10.000 simulações × ~1.800 dias OOS ocuparia ~7GB de RAM — o Colab trava em 12GB. Com lotes de 500, o pico é ~350MB.

## CÉLULA 9 — 🔒 Funções dos gráficos dos testes estatísticos

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
# Gera os gráficos de todos os testes estatísticos e de robustez.

def plotar_testes(res_t, res_boot, res_mc, res_dsr, res_wrc,
                  titulo='Testes Estatísticos e de Robustez'):
    """
    Gera grade 3×2 com os gráficos de todos os testes.

    Parâmetros
    ----------
    res_t    : resultado de teste_t_student()
    res_boot : resultado de bootstrap_sharpe()
    res_mc   : resultado de monte_carlo()
    res_dsr  : resultado de deflated_sharpe()
    res_wrc  : resultado de whites_reality_check()
    titulo   : título geral da figura
    """

    fig, axes = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(titulo, fontsize=13)
    ax = axes.flatten()


    # --------------------------------------------------
    # Gráfico 1 — t-Student: distribuição t e região crítica
    # --------------------------------------------------
    #
    # Mostra a distribuição t com os graus de liberdade da amostra.
    # A área sombreada em vermelho é a região de rejeição (p < 0.05).
    # A linha vertical laranja mostra onde o t_stat do grupo caiu.
    # Se a linha estiver na área vermelha → resultado significativo.

    gl   = res_t['graus_liberdade']
    t_st = res_t['t_stat']
    x    = np.linspace(-5, 5, 500)
    y    = scipy_stats.t.pdf(x, df=gl)

    t_critico = scipy_stats.t.ppf(0.95, df=gl)  # valor crítico unilateral 5%

    ax[0].plot(x, y, color='steelblue', linewidth=2)

    # Área de rejeição (cauda direita)
    x_rej = x[x >= t_critico]
    ax[0].fill_between(x_rej, scipy_stats.t.pdf(x_rej, df=gl),
                       color='red', alpha=0.3, label='Região de rejeição (5%)')

    # Linha do t_stat observado
    ax[0].axvline(t_st, color='darkorange', linewidth=2,
                  label=f't_stat = {t_st:.3f}')
    ax[0].axvline(t_critico, color='red', linewidth=1,
                  linestyle='--', label=f't crítico = {t_critico:.3f}')

    ax[0].set_title(
        f't-Student | p = {res_t["p_valor"]:.3f} | '
        + ('✓ Significativo' if res_t['significativo'] else '✗ Não significativo')
    )
    ax[0].set_xlabel('Valor de t')
    ax[0].set_ylabel('Densidade')
    ax[0].legend(fontsize=8)


    # --------------------------------------------------
    # Gráfico 2 — Bootstrap: distribuição dos Sharpes simulados
    # --------------------------------------------------
    #
    # Histograma dos Sharpes obtidos por reamostagem Bootstrap.
    # Mostra a incerteza ao redor do Sharpe estimado.
    # Quanto mais estreita a distribuição, mais confiável a estimativa.
    # As linhas verticais mostram o IC 95% e o valor original.

    sharpes = res_boot['sharpes_simulados']
    ic_low  = res_boot['ic_95_low']
    ic_high = res_boot['ic_95_high']
    sh_orig = res_boot['sharpe_original']

    ax[1].hist(sharpes, bins=40, edgecolor='white',
               color='steelblue', alpha=0.8)
    ax[1].axvline(sh_orig, color='darkorange', linewidth=2,
                  label=f'Sharpe original: {sh_orig:.3f}')
    ax[1].axvline(ic_low,  color='red', linewidth=1.5, linestyle='--',
                  label=f'IC 95%: [{ic_low:.3f}, {ic_high:.3f}]')
    ax[1].axvline(ic_high, color='red', linewidth=1.5, linestyle='--')
    ax[1].axvline(0, color='gray', linewidth=1, linestyle=':')

    ax[1].set_title(
        f'Bootstrap Sharpe | '
        f'P(Sharpe > 0) = {res_boot["prob_sharpe_pos"]:.1%}'
    )
    ax[1].set_xlabel('Sharpe')
    ax[1].set_ylabel('Frequência')
    ax[1].legend(fontsize=8)


    # --------------------------------------------------
    # Gráfico 3 — Monte Carlo: distribuição de retornos acumulados
    # --------------------------------------------------
    #
    # Histograma dos retornos acumulados de N embaralhamentos aleatórios.
    # A linha laranja mostra onde o retorno real da estratégia se situa.
    # Se estiver à direita da maioria → a ordem dos trades importa.

    if 'erro' not in res_mc:
        ret_sim = res_mc['retornos_simulados']
        ret_r   = res_mc['retorno_real']
        pct_r   = res_mc['percentil_retorno']

        ax[2].hist(ret_sim, bins=40, edgecolor='white',
                   color='steelblue', alpha=0.8)
        ax[2].axvline(ret_r, color='darkorange', linewidth=2,
                      label=f'Retorno real: {ret_r:.1%}')
        ax[2].axvline(np.percentile(ret_sim, 95), color='red',
                      linewidth=1.5, linestyle='--',
                      label='Percentil 95%')
        ax[2].axvline(0, color='gray', linewidth=1, linestyle=':')

        ax[2].xaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax[2].set_title(
            f'Monte Carlo — Retorno Acumulado | '
            f'Supera {pct_r:.0%} das simulações'
        )
        ax[2].set_xlabel('Retorno Acumulado')
        ax[2].set_ylabel('Frequência')
        ax[2].legend(fontsize=8)

    else:
        ax[2].text(0.5, 0.5, res_mc['erro'],
                   ha='center', va='center',
                   transform=ax[2].transAxes, color='gray')
        ax[2].set_title('Monte Carlo — Retorno Acumulado')


    # --------------------------------------------------
    # Gráfico 4 — Monte Carlo: distribuição de drawdowns
    # --------------------------------------------------
    #
    # Mesma lógica, mas para o máximo drawdown.
    # Mostra se o drawdown real é melhor ou pior do que
    # a maioria dos cenários aleatórios.
    # Um drawdown real menos severo que a maioria das
    # simulações indica que a sequência de trades ajuda
    # a controlar as perdas.

    if 'erro' not in res_mc:
        dd_sim = res_mc['drawdowns_simulados']
        dd_r   = res_mc['drawdown_real']
        pct_d  = res_mc['percentil_drawdown']

        ax[3].hist(dd_sim, bins=40, edgecolor='white',
                   color='salmon', alpha=0.8)
        if not np.isnan(dd_r):
            ax[3].axvline(dd_r, color='darkorange', linewidth=2,
                          label=f'Drawdown real: {dd_r:.1%}')
        ax[3].axvline(np.percentile(dd_sim, 5), color='red',
                      linewidth=1.5, linestyle='--',
                      label='Percentil 5%')

        ax[3].xaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax[3].set_title(
            f'Monte Carlo — Máximo Drawdown | '
            f'Melhor que {pct_d:.0%} das simulações'
        )
        ax[3].set_xlabel('Máximo Drawdown')
        ax[3].set_ylabel('Frequência')
        ax[3].legend(fontsize=8)

    else:
        ax[3].text(0.5, 0.5, res_mc['erro'],
                   ha='center', va='center',
                   transform=ax[3].transAxes, color='gray')
        ax[3].set_title('Monte Carlo — Máximo Drawdown')


    # --------------------------------------------------
    # Gráfico 5 — Deflated Sharpe: comparação visual
    # --------------------------------------------------
    #
    # Gráfico de barras comparando o Sharpe observado com o
    # Sharpe esperado pelo acaso (dado o número de testes).
    # Mostra também o DSR como indicador de aprovação/reprovação.

    categorias  = ['Sharpe\nobservado', 'Sharpe esperado\npelo acaso']
    valores     = [res_dsr['sharpe_observado'], res_dsr['sharpe_esperado']]
    cores_barra = ['steelblue', 'salmon']

    barras = ax[4].bar(categorias, valores, color=cores_barra,
                       edgecolor='white', width=0.4)
    ax[4].axhline(0, color='gray', linestyle='--', linewidth=0.8)

    # Adiciona os valores em cima de cada barra
    for barra, val in zip(barras, valores):
        ax[4].text(barra.get_x() + barra.get_width() / 2,
                   val + 0.01,
                   f'{val:.3f}',
                   ha='center', va='bottom', fontsize=10)

    # Adiciona o DSR como anotação
    cor_dsr = 'green' if res_dsr['aprovado'] else 'red'
    ax[4].text(0.5, 0.05,
               f'DSR = {res_dsr["dsr"]:.3f} | n_testes = {res_dsr["n_testes"]}',
               ha='center', transform=ax[4].transAxes,
               fontsize=9, color=cor_dsr)

    ax[4].set_title(
        'Deflated Sharpe Ratio | ' +
        ('✓ Aprovado (DSR > 0.95)' if res_dsr['aprovado']
         else '✗ Sharpe possivelmente inflado')
    )
    ax[4].set_ylabel('Sharpe Ratio')


    # --------------------------------------------------
    # Gráfico 6 — White's Reality Check: distribuição nula
    # --------------------------------------------------
    #
    # Histograma da distribuição nula (máximos Bootstrap).
    # A linha laranja mostra o melhor excesso real sobre o benchmark.
    # Se estiver à direita do percentil 95% → rejeita H0.
    # Se estiver dentro da distribuição → resultado pode ser sorte.

    maximos = res_wrc['maximos_bootstrap']
    melhor  = res_wrc['melhor_excesso']
    p_wrc   = res_wrc['p_valor']
    p95     = np.percentile(maximos, 95)

    ax[5].hist(maximos, bins=40, edgecolor='white',
               color='steelblue', alpha=0.8,
               label='Distribuição nula (Bootstrap)')
    ax[5].axvline(melhor, color='darkorange', linewidth=2,
                  label=f'Melhor estratégia: {melhor:.5f}')
    ax[5].axvline(p95, color='red', linewidth=1.5, linestyle='--',
                  label=f'Percentil 95%: {p95:.5f}')

    ax[5].set_title(
        f"White's Reality Check | p = {p_wrc:.3f} | "
        + ('✓ Rejeita H0' if res_wrc['aprovado'] else '✗ Não rejeita H0')
    )
    ax[5].set_xlabel('Excesso médio sobre buy-and-hold (por dia)')
    ax[5].set_ylabel('Frequência')
    ax[5].legend(fontsize=8)


    plt.tight_layout()
    plt.show()


def imprimir_testes(res_t, res_boot, res_mc, res_dsr, res_wrc):
    """
    Imprime um resumo textual de todos os testes no console.
    """
    print('=' * 60)
    print('  TESTES ESTATÍSTICOS E DE ROBUSTEZ')
    print('=' * 60)

    print('\n--- t-STUDENT ---')
    print(f'  t_stat          : {res_t["t_stat"]:.4f}')
    print(f'  p-valor         : {res_t["p_valor"]:.4f}')
    print(f'  IC 95% retorno  : [{res_t["ic_95_low"]:.2%}, {res_t["ic_95_high"]:.2%}]')
    print(f'  Conclusão       : {res_t["conclusao"]}')

    print('\n--- BOOTSTRAP DO SHARPE ---')
    print(f'  Sharpe original : {res_boot["sharpe_original"]:.3f}')
    print(f'  IC 95%          : [{res_boot["ic_95_low"]:.3f}, {res_boot["ic_95_high"]:.3f}]')
    print(f'  P(Sharpe > 0)   : {res_boot["prob_sharpe_pos"]:.1%}')

    print('\n--- MONTE CARLO ---')
    if 'erro' not in res_mc:
        print(f'  {res_mc["conclusao_retorno"]}')
        print(f'  {res_mc["conclusao_drawdown"]}')
    else:
        print(f'  {res_mc["erro"]}')

    print('\n--- DEFLATED SHARPE RATIO ---')
    print(f'  {res_dsr["conclusao"]}')

    print("\n--- WHITE'S REALITY CHECK ---")
    print(f'  {res_wrc["conclusao"]}')

## CÉLULA 10 — 🔒 Roda tudo e plota os gráficos

In [ ]:
# 🔒 NÃO ALTERE ESTA CÉLULA
# Execute esta célula para rodar o walk-forward completo
# e ver todos os resultados do seu grupo.

# Passo 1: roda o walk-forward com o setup do grupo
records = rodar_walk_forward(
    data         = data,
    years        = years,
    train_years  = train_years,
    test_years   = test_years,
    gerar_grade  = gerar_grade,
    calcular_sinal = calcular_sinal,
    aplicar_stop = aplicar_stop,
    custo_trade  = CUSTO_TRADE
)

# Passo 2: calcula todas as métricas
m = calcular_metricas(records)


# Execute esta célula para rodar todos os testes estatísticos.
# Pode demorar ~30 segundos por causa das simulações.

print('Rodando testes estatísticos...')
print('(Bootstrap + Monte Carlo: 1.000 simulações cada)\n')

# ------ t-Student ------
res_t = teste_t_student(m)

# ------ Bootstrap do Sharpe ------
res_boot = bootstrap_sharpe(m)

# ------ Monte Carlo ------
res_mc = monte_carlo(m)

# ------ Deflated Sharpe Ratio ------
# n_testes = total de combinações testadas na grade do grupo.
# len(gerar_grade()) calcula isso automaticamente.
n_testes = len(gerar_grade())
res_dsr  = deflated_sharpe(m, n_testes=n_testes)

# ------ White's Reality Check ------
# Este teste recalcula TODAS as combinações no período OOS —
# por isso demora um pouco mais que os outros.
res_wrc = whites_reality_check(
    data         = data,
    years        = years,
    train_years  = train_years,
    test_years   = test_years,
    gerar_grade  = gerar_grade,
    calcular_sinal = calcular_sinal,
    custo_trade  = CUSTO_TRADE
)

# Passo 3: imprime o resumo
imprimir_metricas(m)

# ------ Imprimir resumo textual ------
imprimir_testes(res_t, res_boot, res_mc, res_dsr, res_wrc)

# ------ Gráficos ------
titulo_testes = f'{NOME_DO_GRUPO} — {ATIVO} — {DESCRICAO}'
plotar_testes(res_t, res_boot, res_mc, res_dsr, res_wrc,
              titulo=titulo_testes)

# Passo 4: gera os gráficos
titulo = f'{NOME_DO_GRUPO} — {ATIVO} — {DESCRICAO}'
plotar_resultados(m, titulo=titulo)
